### View und Controller für unser Spiel mit Bullets und Kanone
Um bequemer Testen zu können, erstellen wird noch eine View und einen Controller.
Geschossen wird mit Space, der Kanonenwinkel wird mit LeftArrow/Rightarrow geändert,
und in einer ersten Version werden die Bullets mit `ArrowUp` bewegt.


In einer nächsten Version wird dies dann von einem Async-Task erledigt, welcher z.B. in `new_game` gestartet werden kann:
```python
def __init__(self):
    self.is_running = False

def new_game(self):
    if not self.is_running:
        self._run()

def stop(self):
   self.is_running = False

def step(self):
    '''bewegt die Bullers'''

def _run(self):
    async def move_bullets():
        while self.is_running:
            self.step()
            await asyncio.sleep(0.2)

    self.is_running = True
    self.task = asyncio.create_task(move_bullets(), name='move_bullets')
```

**Bemerkung**:  
`asyncio.all_tasks()` liefert ein Objekt mit allen laufenden Tasks,
über welches man mit einem For-Loop iterieren kann, um z.B. einen bestimmten Task zu finden und zu canceln.
```python
for task in asyncio.all_tasks():
    if task.get_name() == 'my_task':
    task.cancel()
```

In [5]:
import importlib
import shooter
from model_view_controller import BaseView
from ipycanvas import hold_canvas
importlib.reload(shooter)


class View(BaseView):
    def __init__(self, game):  # erstellt Default-View mit width=height=100 und debug=True
        super().__init__(game)
        self.redraw()

    def redraw(self):
        with hold_canvas(self.canvas):
            self.canvas.clear()
            self.game.cannon.draw(self.canvas)
            for bullet in self.game.bullets:
                bullet.draw(self.canvas)

    def update(self, event, data):
        self.redraw()


game = shooter.Game()
game.new_game()
view = View(game)
view

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [6]:
game.shoot()
game.decrease_angle()
game.shoot()

In [3]:
game.step()

In [8]:
from model_view_controller import Controller


game = shooter.Game()

callbacks = {'n': game.new_game,
             ' ': game.shoot,
             'ArrowLeft': game.decrease_angle,
             'ArrowRight': game.increase_angle,
             'ArrowUp': game.step,
             }

game.new_game()
view = View(game)
controller = Controller(game, view, callbacks)
controller

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

### Aufgaben
1. Bewege die Bullets mit einem Async-Task (siehe ganz oben).
2. Wir wollen, dass die Bullets an den Wänden reflektiert werden.
   Die Idee ist, dass der Geschwindigkeitsvektor der Bullets in der Methode `_move_bullets` jeweils vor `bullet.move()` wie folgt angepasst      wird:

   Setze `new_pos  = bullet.pos + bullet.speed`  
   Ist eine Komponente von `new_pos` ausserhalb der Leinwand, wird die entsprechende Komponente von `bullet.speed` mit -1 multipliziert.